# 🚀 XAUUSD Autonomous Trading Policy & Position Management ML

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BlamzKunG/XAUUSD-Trading-Policy-ML/blob/main/notebooks/Train_XAUUSD_Trading_Policy_Colab.ipynb)

### **Core Philosophy & Architecture**
1. **Autonomous Trading Policy**: The ML model is not a simple Buy/Sell predictor or room-to-run classifier. It operates as a full closed-loop Trading Policy Agent that decides `BUY`, `SELL`, `HOLD`, `ADD`, `REDUCE`, `CLOSE`, and `REVERSE`, and actively manages positions throughout their lifecycle.
2. **Strict Scale-Invariance**: Zero raw price inputs (Close=1500 vs Close=5000 is completely resolved via stationary ATR-normalized deltas, log returns, and candlestick microstructure).
3. **Counterfactual Teacher Simulator**: Evaluates forward trajectories for multiple hypothetical position states to assign optimal utility-maximizing actions with friction penalties.
4. **Strict Data Hygiene**: Train (2020-2024), Validation (2025), Blind Test (2026 LOCKED).
5. **Direct MetaTrader 5 ONNX Export**: Compiles directly to ONNX Opset 13 (IR Version 8) for native MT5 EA execution.

## 🛠️ Step 1: Environment Setup & GPU Check

In [ ]:
# Check GPU availability
!nvidia-smi

# Install required packages
!pip install -q torch torchvision torchaudio lightgbm onnx onnxruntime matplotlib seaborn pandas numpy

## 📦 Step 2: Clone or Pull Repository

In [ ]:
import os
import sys

REPO_NAME = "XAUUSD-Trading-Policy-ML"
REPO_URL = "https://github.com/BlamzKunG/XAUUSD-Trading-Policy-ML.git"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_NAME}...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
else:
    print(f"Pulling latest updates for {REPO_NAME}...")
    %cd {REPO_NAME}
    !git pull

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print("Working directory:", os.getcwd())

## 📂 Step 3: Load XAUUSD M1 Dataset
Mount Google Drive or upload `XAUUSD.iux_M1_20200102_to_20251230.csv`

In [ ]:
import glob
import pandas as pd

csv_path = None

# Check if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_matches = glob.glob('/content/drive/MyDrive/**/XAUUSD*2025*.csv', recursive=True)
    if drive_matches:
        csv_path = drive_matches[0]
        print(f"Found dataset in Google Drive: {csv_path}")
except Exception as e:
    print("Google Drive not mounted or not found. Checking local folder...")

# Fallback local search
if not csv_path or not os.path.exists(csv_path):
    local_matches = glob.glob('**/XAUUSD*.csv', recursive=True) + glob.glob('/storage/emulated/0/Download/EA/XAUUSD*.csv')
    if local_matches:
        csv_path = local_matches[0]
        print(f"Found local dataset: {csv_path}")
    else:
        print("Please upload your XAUUSD M1 CSV dataset:")
        from google.colab import files
        uploaded = files.upload()
        csv_path = list(uploaded.keys())[0]

print(f"Loading dataset from: {csv_path}...")
df_raw = pd.read_csv(csv_path)
print(f"Loaded {len(df_raw):,} bars! Columns: {list(df_raw.columns)}")
df_raw.head()

## 🛡️ Step 4: Strict Train / Validation Split (2026 LOCKED)

In [ ]:
# Standardize datetime index
if 'datetime' in df_raw.columns:
    df_raw['dt'] = pd.to_datetime(df_raw['datetime'])
elif 'timestamp' in df_raw.columns:
    df_raw['dt'] = pd.to_datetime(df_raw['timestamp'], unit='s')
else:
    df_raw['dt'] = pd.to_datetime(df_raw.index)

df_raw.sort_values('dt', inplace=True)
df_raw.reset_index(drop=True, inplace=True)

# Strict split:
# Train: 2020.01.01 to 2024.12.31
# Validation: 2025.01.01 to 2025.12.31
# 2026 is strictly LOCKED and forbidden
train_mask = (df_raw['dt'] >= '2020-01-01') & (df_raw['dt'] < '2025-01-01')
val_mask = (df_raw['dt'] >= '2025-01-01') & (df_raw['dt'] < '2026-01-01')

df_train = df_raw[train_mask].copy().reset_index(drop=True)
df_val = df_raw[val_mask].copy().reset_index(drop=True)

print(f"Train Data (2020-2024): {len(df_train):,} bars ({df_train['dt'].min()} -> {df_train['dt'].max()})")
print(f"Val Data (2025 Out-of-Sample): {len(df_val):,} bars ({df_val['dt'].min()} -> {df_val['dt'].max()})")

## 🔬 Step 5: Feature Engineering (Scale-Invariant Market Features)

In [ ]:
from scripts.features_policy import extract_market_state_features, MARKET_FEATURE_NAMES, POSITION_FEATURE_NAMES

print("Extracting stationary scale-invariant market features for Train set...")
feat_train, atr_train, close_train = extract_market_state_features(df_train)

print("Extracting stationary scale-invariant market features for Val set...")
feat_val, atr_val, close_val = extract_market_state_features(df_val)

# Drop initial warm-up NaNs (first 200 bars for EMA200)
warmup = 200
feat_train = feat_train.iloc[warmup:].fillna(0.0).reset_index(drop=True)
df_train = df_train.iloc[warmup:].reset_index(drop=True)
atr_train = atr_train.iloc[warmup:].reset_index(drop=True)
close_train = close_train.iloc[warmup:].reset_index(drop=True)

feat_val = feat_val.iloc[warmup:].fillna(0.0).reset_index(drop=True)
df_val = df_val.iloc[warmup:].reset_index(drop=True)
atr_val = atr_val.iloc[warmup:].reset_index(drop=True)
close_val = close_val.iloc[warmup:].reset_index(drop=True)

print(f"Market Features Shape: Train={feat_train.shape}, Val={feat_val.shape}")
print(f"Features ({len(MARKET_FEATURE_NAMES)}): {MARKET_FEATURE_NAMES}")

## 🧠 Step 6: Counterfactual Teacher Simulation (Policy Labeling)

In [ ]:
from scripts.counterfactual_simulator import build_augmented_training_dataset, ACTION_NAMES

# Generate augmented state-action training dataset
# Subsample step = 4 bars to reduce auto-correlation while preserving rich market conditions
X_train, y_act_train, y_sz_train, y_sl_train, y_tp_train = build_augmented_training_dataset(
    market_features=feat_train,
    close_prices=close_train.to_numpy(),
    high_prices=df_train['high'].to_numpy(),
    low_prices=df_train['low'].to_numpy(),
    atr_values=atr_train.to_numpy(),
    horizon=60,
    subsample_step=4
)

print(f"\nAugmented Training Matrix X: {X_train.shape} (Total 40 features: 31 Market + 9 Position)")

## ⚡ Step 7: Train Deep Multi-Head Policy Network (PyTorch / GPU)

In [ ]:
from scripts.train_policy_suite import train_pytorch_policy, export_model_to_onnx, save_policy_metadata

# Train PyTorch Deep Policy Agent
model = train_pytorch_policy(
    X_train=X_train,
    y_action_train=y_act_train,
    y_size_train=y_sz_train,
    y_sl_train=y_sl_train,
    y_tp_train=y_tp_train,
    epochs=30,
    batch_size=512,
    lr=1e-3
)

# Export to ONNX for MetaTrader 5
onnx_path = "models/xauusd_trading_policy.onnx"
export_model_to_onnx(model, onnx_path, input_dim=40, opset_version=13)

# Save Metadata
metadata_path = "models/policy_metadata.json"
save_policy_metadata(
    output_path=metadata_path,
    feature_names=MARKET_FEATURE_NAMES + POSITION_FEATURE_NAMES,
    action_names=ACTION_NAMES,
    mean_stats=np.mean(X_train, axis=0),
    std_stats=np.std(X_train, axis=0)
)

## 📈 Step 8: Closed-Loop Backtesting on 2025 Out-of-Sample Data

In [ ]:
import torch
import matplotlib.pyplot as plt
from scripts.backtest_policy_evaluator import run_closed_loop_backtest

# Create fast PyTorch inference predictor for backtest
model.eval()
device = next(model.parameters()).device

def policy_predictor(state_1x40: np.ndarray):
    with torch.no_grad():
        t_in = torch.tensor(state_1x40, dtype=torch.float32, device=device)
        logits, size, order = model(t_in)
        act_id = int(torch.argmax(logits, dim=-1).item())
        sz = float(size.item())
        sl_atr = float(order[0, 0].item())
        tp_atr = float(order[0, 1].item())
    return act_id, sz, sl_atr, tp_atr

print("Running Closed-Loop Backtest on 2025 Out-of-Sample Validation Data...")
results = run_closed_loop_backtest(
    df=df_val,
    market_features=feat_val,
    atr_series=atr_val,
    policy_predictor=policy_predictor,
    initial_balance=10000.0,
    lot_base=0.1,
    spread_points=2.0
)

print("\n================ 2025 BACKTEST PERFORMANCE ================")
print(f"Initial Balance:      ${results['initial_balance']:,.2f}")
print(f"Final Equity:         ${results['final_equity']:,.2f}")
print(f"Net Profit:           ${results['net_profit']:,.2f} ({results['return_pct']:.2f}%)")
print(f"Profit Factor:        {results['profit_factor']:.2f}")
print(f"Win Rate:             {results['win_rate']:.2f}%")
print(f"Max Drawdown:         {results['max_drawdown_pct']:.2f}%")
print(f"Total Trades:         {results['total_trades']:,}")
print(f"Avg Trade PnL:        ${results['avg_trade_pnl']:.2f}")
print("===========================================================")

# Plot Equity Curve
plt.figure(figsize=(14, 6))
plt.plot(results['equity_curve'], label='Policy Agent Equity ($)', color='#10b981', linewidth=1.5)
plt.title('2025 Out-of-Sample Closed-Loop Equity Curve (XAUUSD M1)', fontsize=14, fontweight='bold')
plt.xlabel('Time (M1 Bars)', fontsize=12)
plt.ylabel('Account Equity ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('docs/equity_curve_2025.png', dpi=300)
plt.show()

## 📤 Step 9: Automatic Commit & Push ONNX Model to GitHub

In [ ]:
# Push trained ONNX model and metadata to GitHub repository
!git config --global user.name "BlamzKunG"
!git config --global user.email "blamzkung@users.noreply.github.com"

!git add models/ docs/
!git commit -m "feat(model): add trained XAUUSD Trading Policy ONNX and metadata [skip ci]"
!git push origin main
print("🎉 Successfully pushed trained models to GitHub repo!")